In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, box

In [ ]:
desired_regions = snakemake.params.desired_regions

In [ ]:
geodata_files = {
    "onshore": snakemake.input.euroshape,
}

In [ ]:
# Onshore shape

In [ ]:
nuts_lvl = 2

euro_onshore = (
    gpd.read_file(snakemake.input.euroshape)
    .replace({"GB": "UK", "EL": "GR"})
    .query("LEVL_CODE == @nuts_lvl & CNTR_CODE in @desired_regions ")
    .rename(columns={"NUTS_ID": "index"})
    .loc[:, ["index", "CNTR_CODE", "geometry"]]
    .set_index(["index"])
)

In [ ]:
rectx1 = -12
rectx2 = 44
recty1 = 33
recty2 = 81


polygon = Polygon(
    [
        (rectx1, recty1),
        (rectx1, recty2),
        (rectx2, recty2),
        (rectx2, recty1),
        (rectx1, recty1),
    ]
)

euro_onshore = gpd.clip(euro_onshore, polygon)

euro_onshore.plot()

In [ ]:
# Remove Svalbard

euro_onshore = gpd.clip(euro_onshore, box(rectx1, recty1, rectx2, 72))

euro_onshore.plot()

In [ ]:
# Remove Jan Mayen

nor = euro_onshore.query("CNTR_CODE =='NO'").clip(box(0, recty1, rectx2, recty2))

euro_onshore = pd.concat([euro_onshore.query("CNTR_CODE != 'NO'"), nor])

euro_onshore.plot()

In [ ]:
euro_onshore.to_file(snakemake.output.onshoreshape)